# NB01 – Primeiro voo: onde compensa abastecer com etanol?
**Métodos e Ferramentas de Data Science – FGV MBA · Aula 1**

## A pergunta de negócio
Uma empresa tem uma frota de carros **flex** espalhada pelo Brasil. O gerente de frota quer saber:

> **Em quais estados compensa abastecer com etanol em vez de gasolina?**

Regra prática do mercado: o etanol rende cerca de 70% da gasolina. Por isso, **compensa usar etanol quando o preço dele é até 70% do preço da gasolina**.

## Os dados
Levantamento semanal de preços de combustíveis da **ANP** (Agência Nacional do Petróleo), 1º semestre de 2026.
Fonte: [ANP – Série histórica de preços de combustíveis](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/serie-historica-de-precos-de-combustiveis).
Para a aula, usamos um recorte com 8 colunas, sem endereço nem CNPJ dos postos.

## Como usar
Execute as células **de cima para baixo**, uma por vez, com **Shift + Enter**. Leia o resultado de cada uma antes de seguir.

## 1. Ferramentas
Em Python, cada **biblioteca** é uma caixa de ferramentas. `pandas` trabalha com tabelas; `matplotlib` e `seaborn` desenham gráficos.
O `as pd` é só um apelido para escrever menos.

In [ ]:
import pandas as pd               # tabelas
import matplotlib.pyplot as plt   # gráficos
import seaborn as sns             # gráficos com visual pronto

pd.set_option("display.float_format", "{:.2f}".format)   # mostra números com 2 casas

## 2. Carregar os dados
A linha abaixo baixa o arquivo direto do repositório da disciplina e cria uma **tabela** (no pandas, chamada **DataFrame**) com o nome `df`.

> **Deu erro de conexão?** Veja o *Plano B* no fim do notebook.

In [ ]:
URL = "https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/dados/anp_combustiveis_2026s1.csv.gz"
df = pd.read_csv(URL)    # lê o arquivo e cria a tabela
print("Dados carregados!")

## 3. Primeira olhada
Toda análise começa olhando os dados. `shape` mostra **(linhas, colunas)**; `head()` mostra as primeiras linhas.

In [ ]:
print("Linhas e colunas:", df.shape)
df.head()

Cada linha é **uma coleta de preço em um posto, em uma data**. Colunas:

| Coluna | O que é |
|---|---|
| `regiao` | Sigla da região (N, NE, CO, SE, S) |
| `uf` | Sigla do estado |
| `municipio` | Município do posto |
| `produto` | Combustível |
| `data_coleta` | Data da coleta |
| `preco_venda` | Preço na bomba |
| `unidade` | Unidade do preço |
| `bandeira` | Distribuidora do posto |

In [ ]:
# Quantas coletas temos de cada produto?
df["produto"].value_counts()

**Atenção:** o GNV é vendido por metro cúbico (`R$ / m³`), os demais por litro. Comparar preços de unidades diferentes é um erro clássico.
Ficamos só com **gasolina comum** e **etanol**, que respondem à nossa pergunta.

In [ ]:
# Filtra: mantém só as linhas em que o produto é GASOLINA ou ETANOL
flex = df[df["produto"].isin(["GASOLINA", "ETANOL"])]
print("Coletas de gasolina e etanol:", len(flex))

## 4. Preço médio por estado
`groupby` agrupa as linhas por categoria e calcula um resumo de cada grupo, como uma **tabela dinâmica** do Excel.

In [ ]:
# Preço médio de cada produto em cada estado
precos = flex.groupby(["uf", "produto"])["preco_venda"].mean().unstack()
precos.head()

In [ ]:
# Razão etanol / gasolina: se for 0,70 ou menos, compensa etanol
precos["razao"] = precos["ETANOL"] / precos["GASOLINA"]
precos["compensa_etanol"] = precos["razao"] <= 0.70
precos.sort_values("razao")

**Pare e pense:** quantos estados passam na regra dos 70%? Eles têm algo em comum?

## 5. O gráfico que responde à pergunta

In [ ]:
ordem = precos.sort_values("razao")
cores = ["#2f7d4f" if c else "#b4443a" for c in ordem["compensa_etanol"]]

plt.figure(figsize=(12, 4))
plt.bar(ordem.index, ordem["razao"], color=cores)
plt.axhline(0.70, color="black", linestyle="--", label="Limite de 70%")
plt.title("Preço do etanol como fração do preço da gasolina, por estado (1º sem. 2026)")
plt.ylabel("Etanol ÷ gasolina")
plt.ylim(0.5, 1.0)
plt.legend()
plt.show()

## 6. Seu estado
Agora é com você. Troque a sigla abaixo pela do estado que quiser analisar e execute.

> Na **Tarefa 1**, você vai usar o estado definido pela sua matrícula.

In [ ]:
ESTADO = "__"   # SEU CÓDIGO AQUI: troque pela sigla do estado, por exemplo "SP"

meu = flex[flex["uf"] == ESTADO]
print(f"Coletas em {ESTADO}:", len(meu))
print(f"Gasolina média: R$ {precos.loc[ESTADO, 'GASOLINA']:.2f}")
print(f"Etanol médio:   R$ {precos.loc[ESTADO, 'ETANOL']:.2f}")
print(f"Razão: {precos.loc[ESTADO, 'razao']:.3f}  →  compensa etanol? {precos.loc[ESTADO, 'compensa_etanol']}")

### Como o preço evoluiu mês a mês no seu estado?

In [ ]:
# Cria a coluna "mes" a partir da data (ex.: 2026-03)
meu = meu.assign(mes=meu["data_coleta"].str[:7])
mensal = meu.groupby(["mes", "produto"])["preco_venda"].mean().unstack()
mensal

In [ ]:
# SEU CÓDIGO AQUI: complete com o nome da tabela mensal que criamos acima
____.plot(marker="o", figsize=(8, 4), title=f"Preço médio mensal em {ESTADO}")
plt.ylabel("R$ por litro")
plt.show()

## 7. Antes de levar à diretoria: limitações
Uma boa análise diz o que **não** sabe. Discuta com a turma:

1. **Média simples de coletas:** cada posto pesa igual, não importa quanto vende. Um posto pequeno pesa o mesmo que um grande.
2. **Amostra:** a ANP pesquisa uma amostra de postos e municípios, não todos.
3. **Média do estado esconde diferenças** entre capital e interior, e entre rodovia e cidade.
4. **A regra dos 70% é aproximada:** o rendimento real depende do motor e do uso.
5. **Preço não é custo total:** a frota pode ter contratos, descontos e rotas que mudam a conta.

> Dado não é decisão. O dado informa a decisão, e as limitações dizem quanto confiar nele.

## Erros comuns e como resolver

| Mensagem | O que aconteceu | O que fazer |
|---|---|---|
| `NameError: name 'pd' is not defined` | Você pulou a célula do `import` | Execute as células desde o início (*Ambiente de execução → Executar anteriores*) |
| `NameError: name 'flex' is not defined` | Uma célula anterior não foi executada | Mesma solução acima |
| `KeyError: '__'` ou `KeyError: 'Rj'` | Sigla do estado errada | Use 2 letras maiúsculas, entre aspas: `"RJ"` |
| `HTTPError` ou `URLError` na célula 2 | Sem internet ou rede bloqueada | Use o Plano B abaixo |
| Nada acontece / célula com `[*]` | A célula ainda está rodando | Aguarde alguns segundos |

## Plano B: se o download falhar
1. Baixe o arquivo pelo link do ECLASS (ou peça ao colega): `anp_combustiveis_2026s1.csv.gz`.
2. No Colab, clique no ícone de **pasta** na barra lateral esquerda e arraste o arquivo para lá.
3. Execute a célula abaixo **no lugar da célula 2** e siga normalmente.

In [ ]:
# PLANO B: só execute se o download da célula 2 falhou
# df = pd.read_csv("anp_combustiveis_2026s1.csv.gz")
# print("Dados carregados do arquivo local!")